In [ ]:
import json


class Cluster:
    def __init__(self, representative, members):
        self.representative = representative
        self.members = members

    def __repr__(self):
        return (
            f"Cluster(representative='{self.representative}', members={self.members})"
        )


def clean_name(name):
    return name.split("/")[-1].split(".")[0]


def load_clusters(array):
    result = []
    for obj in array:
        representative = clean_name(obj["representative"])
        members = list(map(clean_name, obj["members"]))
        result.append(Cluster(representative, members))
    return result


def load_all_clusters():
    for mode in ["approximate", "exact"]:
        for method in [
            "hierarchical",
            "affinity-propagation",
            "facility-location",
        ]:
            path = f"{mode}-{method}.json"
            with open(path) as f:
                data = json.load(f)
                yield (mode, method, load_clusters(data["clustering"]["clusters"]))


clustering = {}
for mode, method, clusters in load_all_clusters():
    clustering[(mode, method)] = clusters

assert len(clustering[("approximate", "hierarchical")]) == 88
assert len(clustering[("approximate", "affinity-propagation")]) == 34
assert len(clustering[("approximate", "facility-location")]) == 33

assert len(clustering[("exact", "hierarchical")]) == 82
assert len(clustering[("exact", "affinity-propagation")]) == 34
assert len(clustering[("exact", "facility-location")]) == 33

clustering[("approximate", "hierarchical")][:10]

In [ ]:
import pandas as pd

df = pd.read_csv("geometric_features.csv")
df

In [ ]:
import itertools

# Generate column names to drop (raw angle values, because we have the sine and cosine of these angles)
columns_to_drop = []

# Drop planar angle columns a{i}{j}{k}
for i, j, k in itertools.combinations(range(8), 3):
    columns_to_drop.append(f"a{i}{j}{k}")

# Drop torsion angle columns t{i}{j}{k}{l}
for i, j, k, l in itertools.combinations(range(8), 4):
    columns_to_drop.append(f"t{i}{j}{k}{l}")

# Drop the columns
df_filtered = df.drop(columns=columns_to_drop)
print(f"Original columns: {len(df.columns)}")
print(f"Filtered columns: {len(df_filtered.columns)}")

In [ ]:
from sklearn.model_selection import train_test_split

X = df_filtered.drop(columns=["source_file", "gnra"])
y = df_filtered["gnra"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f"Shape of X: {X.shape}")
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")

print(f"Shape of y: {y.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

classifiers = {
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(random_state=42, probability=True),
}


def evaluate_classifier(X_train, y_train, X_test, y_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    for name, classifier in classifiers.items():
        classifier.fit(X_train_scaled, y_train)
        y_pred = classifier.predict(X_test_scaled)
        yield name, classifier, scaler, classification_report(y_test, y_pred)


for name, clf, scaler, report in evaluate_classifier(X_train, y_train, X_test, y_test):
    print(f"Classifier: {name}")
    print(report)

In [ ]:
positive = df_filtered[df_filtered["gnra"]]
negative = df_filtered[~df_filtered["gnra"]]
splits = {}

for (mode, method), clusters in clustering.items():
    clusters.sort(key=lambda c: len(c.members))

    positive_test_names = []

    for cluster in clusters:
        positive_test_names.extend([cluster.representative] + cluster.members)
        if len(positive_test_names) >= 0.25 * len(positive):
            break

    positive_train = positive[~positive["source_file"].isin(positive_test_names)]
    positive_test = positive[positive["source_file"].isin(positive_test_names)]

    negative_train, negative_test = train_test_split(
        negative, test_size=0.25, random_state=42
    )

    df_train = pd.concat([positive_train, negative_train])
    df_test = pd.concat([positive_test, negative_test])

    X_train = df_train.drop(columns=["source_file", "gnra"])
    X_test = df_test.drop(columns=["source_file", "gnra"])
    y_train = df_train["gnra"]
    y_test = df_test["gnra"]
    splits[(mode, method)] = (X_train, y_train, X_test, y_test)


In [ ]:
from pickle import dump

for (mode, method), (X_train, y_train, X_test, y_test) in splits.items():
    print(f"Evaluating {mode} {method}...")
    for name, clf, scaler, report in evaluate_classifier(
        X_train, y_train, X_test, y_test
    ):
        print(f"Classifier: {name}")
        print(report)

        model_name = name.lower().replace(" ", "-")
        payload = {
            "classifier_name": name,
            "classifier": clf,
            "scaler": scaler,
            "feature_columns": list(X_train.columns),
            "window_size": 8,
            "positive_label": True,
            "split_mode": mode,
            "split_method": method,
        }

        with open(f"{mode}-{method}-{model_name}.pkl", "wb") as f:
            dump(payload, f)